In [1]:
# Set up SQLite database for Walmart sales analysis

import pandas as pd
import sqlite3

# Load the cleaned Walmart dataset
df = pd.read_csv("/content/walmart_sales_cleaned.csv")

# Create SQLite database connection
conn = sqlite3.connect("walmart_sales.db")

# Load the dataset into a SQL table
df.to_sql(
    "walmart_sales",
    conn,
    if_exists="replace",
    index=False
)

print("SQLite database created successfully.")
print("Rows loaded:", len(df))
print("Columns loaded:", len(df.columns))

SQLite database created successfully.
Rows loaded: 6435
Columns loaded: 12


In [2]:
# SQL Query 1: Preview the Walmart sales table

query = """
SELECT *
FROM walmart_sales
LIMIT 5;
"""

result = pd.read_sql_query(query, conn)

display(result)

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,Year,Month,Month_Name,Year_Month
0,1,2010-02-05,1643690.90,0,42.31,2.572,211.096358,8.106,2010,2,Feb,2010-02
1,1,2010-02-12,1641957.44,1,38.51,2.548,211.242170,8.106,2010,2,Feb,2010-02
2,1,2010-02-19,1611968.17,0,39.93,2.514,211.289143,8.106,2010,2,Feb,2010-02
3,1,2010-02-26,1409727.59,0,46.63,2.561,211.319643,8.106,2010,2,Feb,2010-02
4,1,2010-03-05,1554806.68,0,46.50,2.625,211.350143,8.106,2010,3,Mar,2010-03


In [3]:
# SQL Query 2: Calculate overall sales KPIs

query = """
SELECT
    ROUND(SUM(Weekly_Sales), 2) AS Total_Sales,
    ROUND(AVG(Weekly_Sales), 2) AS Average_Weekly_Sales,
    COUNT(*) AS Total_Records,
    COUNT(DISTINCT Store) AS Number_of_Stores
FROM walmart_sales;
"""

result = pd.read_sql_query(query, conn)

display(result)

,Total_Sales,Average_Weekly_Sales,Total_Records,Number_of_Stores
0,6.737219e+09,1046964.88,6435,45


In [4]:
# SQL Query 3: Find the top 10 stores by total sales

query = """
SELECT
    Store,
    ROUND(SUM(Weekly_Sales), 2) AS Total_Sales
FROM walmart_sales
GROUP BY Store
ORDER BY Total_Sales DESC
LIMIT 10;
"""

result = pd.read_sql_query(query, conn)

display(result)

,Store,Total_Sales
0,20,3.013978e+08
1,4,2.995440e+08
2,14,2.889999e+08
3,13,2.865177e+08
4,2,2.753824e+08
5,10,2.716177e+08
6,27,2.538559e+08
7,6,2.237561e+08
8,1,2.224028e+08
9,39,2.074455e+08


In [5]:
# SQL Query 4: Compare holiday and non-holiday sales

query = """
SELECT
    CASE
        WHEN Holiday_Flag = 1 THEN 'Holiday'
        ELSE 'Non-Holiday'
    END AS Week_Type,

    COUNT(*) AS Number_of_Records,

    ROUND(AVG(Weekly_Sales), 2) AS Average_Weekly_Sales,

    ROUND(SUM(Weekly_Sales), 2) AS Total_Sales

FROM walmart_sales

GROUP BY Holiday_Flag

ORDER BY Average_Weekly_Sales DESC;
"""

result = pd.read_sql_query(query, conn)

display(result)

,Week_Type,Number_of_Records,Average_Weekly_Sales,Total_Sales
0,Holiday,450,1122887.89,5.052996e+08
1,Non-Holiday,5985,1041256.38,6.231919e+09


In [6]:
# SQL Query 5: Calculate the top 5 stores' contribution to total sales

query = """
WITH StoreSales AS (
    SELECT
        Store,
        SUM(Weekly_Sales) AS Store_Total_Sales
    FROM walmart_sales
    GROUP BY Store
),

TopFive AS (
    SELECT
        Store,
        Store_Total_Sales
    FROM StoreSales
    ORDER BY Store_Total_Sales DESC
    LIMIT 5
)

SELECT
    ROUND(SUM(Store_Total_Sales), 2) AS Top_5_Sales,
    ROUND(
        SUM(Store_Total_Sales) * 100.0 /
        (SELECT SUM(Weekly_Sales) FROM walmart_sales),
        2
    ) AS Top_5_Percentage
FROM TopFive;
"""

result = pd.read_sql_query(query, conn)

display(result)

,Top_5_Sales,Top_5_Percentage
0,1.451842e+09,21.55


In [7]:
# SQL Query 6: Analyze monthly sales trends

query = """
SELECT
    Year_Month,
    ROUND(SUM(Weekly_Sales), 2) AS Monthly_Total_Sales,
    ROUND(AVG(Weekly_Sales), 2) AS Average_Store_Week_Sales,
    COUNT(*) AS Number_of_Records
FROM walmart_sales
GROUP BY Year_Month
ORDER BY Year_Month;
"""

result = pd.read_sql_query(query, conn)

display(result)

,Year_Month,Monthly_Total_Sales,Average_Store_Week_Sales,Number_of_Records
0,2010-02,1.903330e+08,1057405.46,180
1,2010-03,1.819198e+08,1010665.57,180
2,2010-04,2.314124e+08,1028499.41,225
3,2010-05,1.867109e+08,1037282.97,180
4,2010-06,1.922462e+08,1068034.29,180
5,2010-07,2.325801e+08,1033689.45,225
6,2010-08,1.876401e+08,1042445.06,180
7,2010-09,1.772679e+08,984821.65,180
8,2010-10,2.171618e+08,965163.66,225
9,2010-11,2.028534e+08,1126963.17,180


In [8]:
# SQL Query 7: Rank stores by total sales using a window function

query = """
WITH StoreSales AS (
    SELECT
        Store,
        SUM(Weekly_Sales) AS Total_Sales
    FROM walmart_sales
    GROUP BY Store
)

SELECT
    Store,
    ROUND(Total_Sales, 2) AS Total_Sales,
    RANK() OVER (
        ORDER BY Total_Sales DESC
    ) AS Sales_Rank
FROM StoreSales
ORDER BY Sales_Rank;
"""

result = pd.read_sql_query(query, conn)

display(result.head(10))

,Store,Total_Sales,Sales_Rank
0,20,3.013978e+08,1
1,4,2.995440e+08,2
2,14,2.889999e+08,3
3,13,2.865177e+08,4
4,2,2.753824e+08,5
5,10,2.716177e+08,6
6,27,2.538559e+08,7
7,6,2.237561e+08,8
8,1,2.224028e+08,9
9,39,2.074455e+08,10


In [9]:
# SQL Query 8: Calculate month-over-month sales change using LAG()

query = """
WITH MonthlySales AS (
    SELECT
        Year_Month,
        SUM(Weekly_Sales) AS Monthly_Sales
    FROM walmart_sales
    GROUP BY Year_Month
),

MonthlyComparison AS (
    SELECT
        Year_Month,
        Monthly_Sales,

        LAG(Monthly_Sales) OVER (
            ORDER BY Year_Month
        ) AS Previous_Month_Sales

    FROM MonthlySales
)

SELECT
    Year_Month,

    ROUND(Monthly_Sales, 2) AS Monthly_Sales,

    ROUND(Previous_Month_Sales, 2) AS Previous_Month_Sales,

    ROUND(
        (Monthly_Sales - Previous_Month_Sales)
        * 100.0 / Previous_Month_Sales,
        2
    ) AS MoM_Growth_Percentage

FROM MonthlyComparison

ORDER BY Year_Month;
"""

result = pd.read_sql_query(query, conn)

display(result)

,Year_Month,Monthly_Sales,Previous_Month_Sales,MoM_Growth_Percentage
0,2010-02,1.903330e+08,NaN,NaN
1,2010-03,1.819198e+08,1.903330e+08,-4.42
2,2010-04,2.314124e+08,1.819198e+08,27.21
3,2010-05,1.867109e+08,2.314124e+08,-19.32
4,2010-06,1.922462e+08,1.867109e+08,2.96
5,2010-07,2.325801e+08,1.922462e+08,20.98
6,2010-08,1.876401e+08,2.325801e+08,-19.32
7,2010-09,1.772679e+08,1.876401e+08,-5.53
8,2010-10,2.171618e+08,1.772679e+08,22.50
9,2010-11,2.028534e+08,2.171618e+08,-6.59
